In [1]:
import requests
import uuid
import json
import os

def get_token(auth_token, scope='GIGACHAT_API_PERS'):
    rq_uid = str(uuid.uuid4())

    url = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"

    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
        'Accept': 'application/json',
        'RqUID': rq_uid,
        'Authorization': f'Basic {auth_token}'
    }

    payload = {
        'scope': scope
    }

    try:
        response = requests.post(url, headers=headers, data=payload, verify=False)
        return response
    except requests.RequestException as e:
        print(f"Ошибка: {str(e)}")
        return -1

In [2]:
key = 'MDE5YWY0ZjctMjMxOS03N2ZkLThkOWEtNThiNWZkNTFkODA0OjNiNjViZTUxLTljYjMtNGViYS1hMTA1LTYyZDlmZDdiYTczMQ=='

response = get_token(key)
if response != 1:
  print(response.text)
  giga_token = response.json()['access_token']

{"access_token":"eyJjdHkiOiJqd3QiLCJlbmMiOiJBMjU2Q0JDLUhTNTEyIiwiYWxnIjoiUlNBLU9BRVAtMjU2In0.DFEZM-2UCiJQqJEN3_OPvvwmljSS5Ex1tA9--xGmZB8GihPpHGu0gD7IVfGiwqBkx2H46Khnvtl7f6EBiXEyw7qkOwQfBYC2_rWV9Pb2S6o2TKNvBMjijNdSRileI8-EqYarPXwNkHGMDgqIXuZtpEXHwG70HeCgWJdkdH0DaRR8kJtWeimNcpM-92REoUSTCsByhRqE2n6u4aL-4HcNdWCs58V8SsvIffMt1dN7YmbReAbMSDD5WskZby9MvRuji1OfD9yV56glh8KL6ggVMm1DxZCfRNKVW1PRaxGmMxoinEQbHAkOGvcA0gVckCy5XJbnti51KwTqC_yNVrV75Q.z--OlDLGKk8sHBvb8Y6PQg.GW4-WUtsTEUU-1jV1WyGw2Fs9QXVGZS6b3VCiDB6PB7xEUtS2s5PI4V9xXjhJyjmKWbfMPEzrTpybe683pCbX_bQVXkHvF2US3y38kVcgsUTk5NE2WaXUcODRMi_yx1idYMeYt-B7mmbTjZKbzEH6YZZLeq2gQxpiLJoD3Q0e30ubpbpr9PKGGk9MtxylPV_9czMwf2t0R03QpMVvJxeEWNwIWWR5jdWxSBFo8MUohqnBihpwa87TymTDUQfyYrnY0lXoq-c86lF2afVdE_4tXHcykDZX92z2tmwQEEAF7hgdhmfxznjvJjvXS3kUabRzRPFWwDF_e9jP1l5VItqgWtp9PAG2xAeE7l1FUtmO4b9gBcfE24ROMO4jzyqpRQjGFhFdRsXP75mt1iYylt66-Pmjby9bqD-mcXzgVNNsNgWGuo7vhIjRqxcLzgYATlxvKBgEYd5zyn8kFyS9Vj18zuqGV9dBkFzItrS3Y-Bvg5LXtgede3dY5VPmR9BdMJpmVCWHYBkbky3MzGGA-JSK74qjUmIl

/Users/leonid/anaconda3/lib/python3.11/site-packages/urllib3/connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ngw.devices.sberbank.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [3]:
def get_chat_completion(auth_token, user_message):
    url = "https://gigachat.devices.sberbank.ru/api/v1/chat/completions"

    payload = json.dumps({
        "model": "GigaChat", 
        "messages": [
            {
                "role": "user", 
                "content": user_message  
            }
        ],
        "temperature": 1, 
        "top_p": 0.1,  
        "n": 1,  
        "stream": False,  
        "max_tokens": 512,
        "repetition_penalty": 1,  
        "update_interval": 0 
    })

    headers = {
        'Content-Type': 'application/json',  
        'Accept': 'application/json',  
        'Authorization': f'Bearer {auth_token}' 
    }

    try:
        response = requests.request("POST", url, headers=headers, data=payload, verify=False)
        return response
    except requests.RequestException as e:
        print(f"Произошла ошибка: {str(e)}")
        return -1

In [151]:
# answer = get_chat_completion(giga_token, 'Что такое питон?')

# print(answer.json()['choices'][0]['message']['content'])

In [4]:
# !ls Downloads/tbank_knowledge_2/

def clean_text(text):
    if not text:
        return ""
    text = text.replace('\xa0', ' ').replace('\u200b', ' ')
    return ' '.join(text.split())

In [5]:
folder_path = 'Downloads/tbank_knowledge_2/'

for i in os.listdir(folder_path):

    with open(f'Downloads/tbank_knowledge_2/{i}', 'r', encoding='utf-8') as file:
        data = json.load(file)
        
    if type(data) == list:
        title = data[0]['title']
        content = data[0]['content'][:20000]
        clean_content = clean_text(content)
        clean_title = clean_text(title)
    else:
        title = data['title']
        content = data['content'][:20000]
        clean_content = clean_text(content)
        clean_title = clean_text(title)
    
    if os.path.exists(f'Q_A_articles/{clean_title}.json') is False:

        answer = get_chat_completion(giga_token, f'Напиши пять вопросов и ответов на них к статье {clean_title}: {clean_content}')

        with open(f'Q_A_articles/{clean_title}.json', 'w', encoding='utf-8') as f:
            json.dump(answer.json()['choices'][0]['message'], f, ensure_ascii=False, indent=2)